# Celeb-DF-v3 Dataset Overview

Запускать на сервере `thesis` (путь к датасету прописан в ячейке Setup).

Что показывает этот ноутбук:
- Общее число видео по категориям (Celeb-real, YouTube-real, FaceSwap, FaceReenact, TalkingFace)
- Разбивка по методам синтеза
- Тест-сплит из `List_of_testing_videos.txt`
- Inline-воспроизведение случайных видео из каждой категории

In [ ]:
from pathlib import Path
import random
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Video, HTML, display as idisplay

matplotlib.rcParams["figure.dpi"] = 110

# ---- Путь к датасету на сервере thesis ----
DATA_ROOT = Path("/home/n-salikhova/datasets-retry/extracted/Celeb-DF-v3")
# Если запускаете локально — поменяйте путь выше.

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

print("DATA_ROOT:", DATA_ROOT)
print("exists:   ", DATA_ROOT.exists())

In [ ]:
rows = []
for p in DATA_ROOT.rglob("*"):
    if not (p.is_file() and p.suffix.lower() in VIDEO_EXTS):
        continue
    rel = p.relative_to(DATA_ROOT)
    parts = rel.parts  # e.g. ('Celeb-real', 'id0_0000.mp4')

    top = parts[0]
    if top in ("Celeb-real", "YouTube-real"):
        label, category, method = "real", top, top
    elif top == "Celeb-synthesis" and len(parts) >= 3:
        label = "fake"
        category = parts[1]   # FaceSwap | FaceReenact | TalkingFace
        method   = parts[2]   # BlendFace | DaGAN | EDTalk | …
    else:
        label, category, method = "unknown", top, top

    rows.append({
        "path":          str(p),
        "relative_path": str(rel),
        "filename":      p.name,
        "ext":           p.suffix.lower(),
        "label":         label,
        "category":      category,
        "method":        method,
    })

df = pd.DataFrame(rows)
print(f"Всего видео: {len(df)}")
print(f"  real: {(df['label']=='real').sum()}")
print(f"  fake: {(df['label']=='fake').sum()}")
df.head()

In [ ]:
# ---- Таблица: количество видео по методам ----
summary = (
    df.groupby(["label", "category", "method"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["label", "category", "method"])
)
print("=== Разбивка по методам ===")
display(summary.to_string(index=False))

# ---- Диаграмма ----
fig, ax = plt.subplots(figsize=(12, 7))
pivot = (
    df.groupby(["method", "label"])
    .size()
    .unstack(fill_value=0)
)
order = pivot.sum(axis=1).sort_values().index
pivot = pivot.loc[order]
colors = {"real": "steelblue", "fake": "tomato"}
pivot.plot.barh(
    ax=ax,
    color=[colors.get(c, "grey") for c in pivot.columns],
    stacked=True,
)
ax.set_xlabel("Количество видео")
ax.set_title("Celeb-DF-v3: видео по методу синтеза")
ax.legend(title="label")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Тест-сплит из List_of_testing_videos.txt ----
test_list_path = DATA_ROOT / "List_of_testing_videos.txt"

test_rows = []
with open(test_list_path) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        real_flag, rel_path = line.split(" ", 1)
        parts = Path(rel_path).parts
        if parts[0] in ("Celeb-real", "YouTube-real"):
            category, method = parts[0], parts[0]
        elif parts[0] == "Celeb-synthesis" and len(parts) >= 3:
            category, method = parts[1], parts[2]
        else:
            category = method = parts[0]
        test_rows.append({
            "real":          int(real_flag),
            "relative_path": rel_path,
            "category":      category,
            "method":        method,
        })

test_df = pd.DataFrame(test_rows)
print(f"Тест-сплит: {len(test_df)} видео  |  real={(test_df['real']==1).sum()}  fake={(test_df['real']==0).sum()}")
display(test_df.groupby(["real", "category", "method"]).size().rename("count").reset_index())

In [ ]:
def show_video_sample(label_filter=None, method_filter=None, n=2, width=420):
    """
    Показать n случайных видео из датасета.
    label_filter  : 'real' | 'fake' | None (все)
    method_filter : имя метода ('BlendFace', 'EDTalk', ...) | None (все)
    """
    subset = df.copy()
    if label_filter:
        subset = subset[subset["label"] == label_filter]
    if method_filter:
        subset = subset[subset["method"] == method_filter]
    if subset.empty:
        print("Нет видео для выбранных фильтров.")
        return
    sample = subset.sample(min(n, len(subset)), random_state=None)
    for _, row in sample.iterrows():
        title = f"[{row['label'].upper()}] {row['category']} / {row['method']} — {row['filename']}"
        print(title)
        idisplay(Video(row["path"], embed=True, width=width, html_attributes="controls"))

# ===== Реальные видео (Celeb-real) =====
print("─" * 60)
print("REAL  /  Celeb-real")
show_video_sample(label_filter="real", method_filter="Celeb-real", n=2)

# ===== Реальные видео (YouTube-real) =====
print("─" * 60)
print("REAL  /  YouTube-real")
show_video_sample(label_filter="real", method_filter="YouTube-real", n=2)

# ===== Fake: FaceSwap — BlendFace =====
print("─" * 60)
print("FAKE  /  FaceSwap  /  BlendFace")
show_video_sample(method_filter="BlendFace", n=2)

# ===== Fake: FaceReenact — LivePortrait =====
print("─" * 60)
print("FAKE  /  FaceReenact  /  LivePortrait")
show_video_sample(method_filter="LivePortrait", n=2)

# ===== Fake: TalkingFace — EDTalk =====
print("─" * 60)
print("FAKE  /  TalkingFace  /  EDTalk")
show_video_sample(method_filter="EDTalk", n=2)

In [ ]:
# ---- Сохранить инвентарь видео в CSV ----
OUT_CSV = Path("../metadata/celeb_dfpp_video_inventory.csv").resolve()
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
print(f"Rows: {len(df)}")